# 15 · Matrices e imágenes RGB con Matplotlib

## Matrices

Una **matriz** es una tabla de filas y columnas. En Python se representa como una **lista de listas**: `matriz[fila][columna]`.

In [ ]:
matriz = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

print("Filas:", len(matriz))
print("Columnas:", len(matriz[0]))
print("Elemento (1, 2):", matriz[1][2])
print("Fila 0:", matriz[0])
print("Columna 0:", [fila[0] for fila in matriz])

### Recorrer y construir matrices

In [ ]:
# Recorrer
for i in range(len(matriz)):
    for j in range(len(matriz[i])):
        print(f"[{i}][{j}] = {matriz[i][j]}")

# Sumar todos los elementos
print("Suma:", sum(sum(fila) for fila in matriz))

In [ ]:
# Construir una matriz de 3x4 llena de ceros
filas, columnas = 3, 4
ceros = [[0] * columnas for _ in range(filas)]
ceros[1][2] = 7
print(ceros)

⚠️ **No** uses `[[0] * 4] * 3`: repite la **misma** fila 3 veces (ver cuaderno 11).

In [ ]:
mala = [[0] * 4] * 3
mala[0][0] = 1
print(mala)   # ¡las 3 filas cambiaron!

### Transpuesta (filas ↔ columnas)

In [ ]:
transpuesta = [[matriz[i][j] for i in range(3)] for j in range(3)]
print(transpuesta)

## Imágenes: matrices de colores

Una imagen digital es una matriz de **píxeles**. En **RGB** cada píxel es una tupla de tres valores:
**R**ojo, **G**reen (verde) y **B**lue (azul), de `0` a `255`.

| Color | (R, G, B) |
|-------|-----------|
| Rojo | `(255, 0, 0)` |
| Verde | `(0, 255, 0)` |
| Azul | `(0, 0, 255)` |
| Negro | `(0, 0, 0)` |
| Blanco | `(255, 255, 255)` |
| Amarillo | `(255, 255, 0)` |

Para dibujarlas usamos **Matplotlib** y **NumPy** (que Matplotlib necesita para manejar la matriz). `imshow()` muestra la matriz como imagen.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Imagen de 2 filas x 3 columnas
imagen = [
    [(255, 0, 0), (0, 255, 0), (0, 0, 255)],
    [(255, 255, 0), (255, 255, 255), (0, 0, 0)],
]
imagen = np.array(imagen, dtype=np.uint8)
print(imagen.shape)   # (alto, ancho, canales)

plt.imshow(imagen)
plt.show()

## Crear imágenes con ciclos

Una bandera: ancho 90, alto 60, tres franjas verticales.

In [ ]:
alto, ancho = 60, 90
bandera = []
for fila in range(alto):
    linea = []
    for col in range(ancho):
        if col < ancho // 3:
            linea.append((0, 104, 71))       # verde
        elif col < 2 * ancho // 3:
            linea.append((255, 255, 255))    # blanco
        else:
            linea.append((206, 17, 38))      # rojo
    bandera.append(linea)

plt.imshow(np.array(bandera, dtype=np.uint8))
plt.axis("off")
plt.show()

In [ ]:
# Degradado: el rojo aumenta de izquierda a derecha
alto, ancho = 50, 256
degradado = [[(col, 0, 255 - col) for col in range(ancho)] for _ in range(alto)]
plt.imshow(np.array(degradado, dtype=np.uint8))
plt.axis("off")
plt.show()

## Modificar una imagen

Convertir a **escala de grises**: promedio de R, G y B.

In [ ]:
color = np.array(bandera, dtype=np.uint8)
gris = [[(sum(px) // 3,) * 3 for px in fila] for fila in bandera]

fig, ejes = plt.subplots(1, 2, figsize=(8, 3))
ejes[0].imshow(color);                          ejes[0].set_title("Original")
ejes[1].imshow(np.array(gris, dtype=np.uint8)); ejes[1].set_title("Grises")
for e in ejes:
    e.axis("off")
plt.show()

In [ ]:
# Negativo: 255 - valor
negativo = [[tuple(255 - c for c in px) for px in fila] for fila in bandera]
plt.imshow(np.array(negativo, dtype=np.uint8))
plt.axis("off")
plt.show()

## Leer una imagen real

`plt.imread()` carga un archivo (PNG, JPG) como matriz. Guardemos primero la bandera con `plt.imsave()` y volvamos a abrirla.

In [ ]:
plt.imsave("bandera.png", np.array(bandera, dtype=np.uint8))
leida = plt.imread("bandera.png")
print(leida.shape, leida.dtype)   # PNG se lee como decimales de 0 a 1
print(leida[0][0])                # píxel (0, 0): RGB (+ transparencia)
plt.imshow(leida)
plt.show()

## 🎛️ Mezclador de colores

Mueve los deslizadores para ver cómo se combinan R, G y B.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

r = widgets.IntSlider(value=255, min=0, max=255, description="R", style={"description_width": "20px"})
g = widgets.IntSlider(value=128, min=0, max=255, description="G", style={"description_width": "20px"})
b = widgets.IntSlider(value=0,   min=0, max=255, description="B", style={"description_width": "20px"})
out = widgets.Output()

def pintar(_=None):
    with out:
        out.clear_output(wait=True)
        px = np.full((40, 80, 3), (r.value, g.value, b.value), dtype=np.uint8)
        plt.figure(figsize=(3, 1.5))
        plt.imshow(px)
        plt.axis("off")
        plt.title(f"({r.value}, {g.value}, {b.value})")
        plt.show()

for w in (r, g, b):
    w.observe(pintar, names="value")
display(r, g, b, out)
pintar()

## 🎛️ Pintor de píxeles

Una cuadrícula de 8×8: elige un color y las coordenadas, y pulsa **Pintar**.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

lienzo = [[(255, 255, 255)] * 8 for _ in range(8)]   # matriz 8x8 blanca
# (para evitar el problema de filas compartidas, cada pintura crea una fila nueva)

colores = {"Rojo": (255, 0, 0), "Verde": (0, 160, 0), "Azul": (0, 0, 255),
           "Negro": (0, 0, 0), "Blanco": (255, 255, 255)}
color = widgets.Dropdown(options=colores, description="Color")
fila = widgets.IntSlider(value=0, min=0, max=7, description="fila")
col = widgets.IntSlider(value=0, min=0, max=7, description="columna")
pintar_btn = widgets.Button(description="Pintar", button_style="primary")
out2 = widgets.Output()

def dibujar():
    with out2:
        out2.clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(np.array(lienzo, dtype=np.uint8))
        plt.xticks(range(8)); plt.yticks(range(8))
        plt.show()

def on_pintar(_):
    nueva = list(lienzo[fila.value])   # copiar la fila
    nueva[col.value] = color.value
    lienzo[fila.value] = nueva
    dibujar()

pintar_btn.on_click(on_pintar)
display(widgets.HBox([color, fila, col, pintar_btn]), out2)
dibujar()

## Resumen

- Matriz = lista de listas: `m[fila][columna]`.
- Imagen RGB = matriz de píxeles `(R, G, B)` con valores `0–255`.
- `plt.imshow(np.array(matriz, dtype=np.uint8))` la dibuja.
- Modificar la imagen = modificar los valores de la matriz.